In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully!")

In [ ]:
df = pd.read_csv("../data/retail_sales_data_50000.csv")

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

In [ ]:
missing_values = df.isnull().sum()

missing_values[missing_values > 0]


In [ ]:
df.duplicated().sum()

In [ ]:
df["Order_ID"].duplicated().sum()

In [ ]:
df.describe()

In [ ]:
categorical_columns = [
    "Product_Name",
    "Category",
    "Location",
    "Order_Channel",
    "Payment_Mode"
]

for col in categorical_columns:
    print(f"\n{col}")
    print(df[col].value_counts())

In [ ]:
data_quality = pd.DataFrame({
    "Column": df.columns,
    "Data_Type": df.dtypes.astype(str),
    "Missing_Values": df.isnull().sum().values,
    "Unique_Values": df.nunique().values
})

data_quality

In [ ]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"])

df["Order_Date"].dtype

In [ ]:
df["Year"] = df["Order_Date"].dt.year

df[["Order_Date", "Year"]].head()

In [ ]:

df["Month"] = df["Order_Date"].dt.month

df[["Order_Date", "Year", "Month"]].head()

In [ ]:
df["Month_Name"] = df["Order_Date"].dt.month_name()

df[["Order_Date", "Month", "Month_Name"]].head()

In [ ]:
df["Day_Name"] = df["Order_Date"].dt.day_name()

df[["Order_Date", "Day_Name"]].head()

In [ ]:
df["Calculated_Gross_Sales"] = df["Quantity"] * df["Unit_Price"]

df[[
    "Quantity",
    "Unit_Price",
    "Gross_Sales",
    "Calculated_Gross_Sales"
]].head()

In [ ]:
gross_sales_difference = (
    df["Gross_Sales"] - df["Calculated_Gross_Sales"]
).abs()

gross_sales_difference.max()

In [ ]:
df["Calculated_Net_Sales"] = (
    df["Gross_Sales"] * (1 - df["Discount_Rate"])
)

df[[
    "Gross_Sales",
    "Discount_Rate",
    "Net_Sales",
    "Calculated_Net_Sales"
]].head()

In [ ]:
net_sales_difference = (
    df["Net_Sales"] - df["Calculated_Net_Sales"]
).abs()

net_sales_difference.max()

In [ ]:
df.columns.tolist()

In [ ]:
df.to_csv(
    "../outputs/retail_sales_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

In [ ]:
total_orders = df["Order_ID"].nunique()
total_customers = df["Customer_ID"].nunique()
total_products = df["Product_ID"].nunique()
total_units = df["Quantity"].sum()
total_revenue = df["Net_Sales"].sum()
total_profit = df["Profit"].sum()
average_order_value = df["Net_Sales"].mean()
profit_margin = (total_profit / total_revenue) * 100

print("Total Orders:", total_orders)
print("Total Customers:", total_customers)
print("Total Products:", total_products)
print("Total Units Sold:", total_units)
print("Total Revenue:", total_revenue)
print("Total Profit:", total_profit)
print("Average Order Value:", average_order_value)
print("Profit Margin:", profit_margin)

In [ ]:
kpi_summary = pd.DataFrame({
    "KPI": [
        "Total Orders",
        "Total Customers",
        "Total Products",
        "Total Units Sold",
        "Total Revenue",
        "Total Profit",
        "Average Order Value",
        "Profit Margin"
    ],
    "Value": [
        total_orders,
        total_customers,
        total_products,
        total_units,
        total_revenue,
        total_profit,
        average_order_value,
        profit_margin
    ]
})

kpi_summary

In [ ]:
print(f"Total Revenue: ₹{total_revenue:,.2f}")
print(f"Total Profit: ₹{total_profit:,.2f}")
print(f"Average Order Value: ₹{average_order_value:,.2f}")
print(f"Profit Margin: {profit_margin:.2f}%")

In [ ]:
kpi_summary.to_csv(
    "../outputs/kpi_summary.csv",
    index=False
)

print("KPI summary saved successfully!")

In [ ]:
customer_summary = (
    df.groupby("Customer_ID")
    .agg(
        Total_Orders=("Order_ID", "nunique"),
        Total_Units=("Quantity", "sum"),
        Total_Spent=("Net_Sales", "sum"),
        Total_Profit=("Profit", "sum"),
        Average_Order_Value=("Net_Sales", "mean")
    )
    .reset_index()
)

customer_summary.head()

In [ ]:
top_customers = (
    customer_summary
    .sort_values("Total_Spent", ascending=False)
    .head(10)
)

top_customers

In [ ]:
top_customers = (
    customer_summary
    .sort_values("Total_Spent", ascending=False)
    .head(10)
)

top_customers

In [ ]:
customer_summary["Customer_Type"] = np.where(
    customer_summary["Total_Orders"] == 1,
    "One-Time Customer",
    "Repeat Customer"
)

customer_summary["Customer_Type"].value_counts()

In [ ]:
customer_type_summary = (
    customer_summary
    .groupby("Customer_Type")
    .agg(
        Customer_Count=("Customer_ID", "count"),
        Total_Revenue=("Total_Spent", "sum"),
        Average_Customer_Spend=("Total_Spent", "mean")
    )
    .reset_index()
)

customer_type_summary

In [ ]:
customer_summary["Spend_Rank"] = (
    customer_summary["Total_Spent"]
    .rank(method="min", ascending=False)
    .astype(int)
)

customer_summary.sort_values("Spend_Rank").head(10)

In [ ]:
customer_summary["Spend_Decile"] = pd.qcut(
    customer_summary["Total_Spent"],
    10,
    labels=False
) + 1

In [ ]:
customer_summary["Customer_Segment"] = np.where(
    customer_summary["Spend_Decile"] == 1,
    "Low Value",
    np.where(
        customer_summary["Spend_Decile"].isin([8, 9, 10]),
        "High Value",
        "Medium Value"
    )
)

In [ ]:
customer_summary["Customer_Segment"] = np.where(
    customer_summary["Total_Spent"] >= customer_summary["Total_Spent"].quantile(0.90),
    "High Value",
    np.where(
        customer_summary["Total_Spent"] >= customer_summary["Total_Spent"].quantile(0.60),
        "Medium Value",
        "Low Value"
    )
)

customer_summary["Customer_Segment"].value_counts()

In [ ]:
segment_summary = (
    customer_summary
    .groupby("Customer_Segment")
    .agg(
        Customer_Count=("Customer_ID", "count"),
        Revenue=("Total_Spent", "sum"),
        Profit=("Total_Profit", "sum"),
        Average_Customer_Spend=("Total_Spent", "mean")
    )
    .reset_index()
)

segment_summary

In [ ]:
segment_summary.sort_values("Revenue", ascending=False)

In [ ]:
customer_summary.to_csv(
    "../outputs/customer_analysis.csv",
    index=False
)

segment_summary.to_csv(
    "../outputs/customer_segments.csv",
    index=False
)

print("Customer analysis files saved successfully!")

In [ ]:
category_summary = (
    df.groupby("Category")
    .agg(
        Total_Orders=("Order_ID", "nunique"),
        Units_Sold=("Quantity", "sum"),
        Revenue=("Net_Sales", "sum"),
        Profit=("Profit", "sum")
    )
    .reset_index()
)

category_summary["Profit_Margin"] = (
    category_summary["Profit"] /
    category_summary["Revenue"] * 100
)

category_summary.sort_values(
    "Revenue",
    ascending=False
)

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    data=category_summary.sort_values("Revenue", ascending=False),
    x="Category",
    y="Revenue"
)

plt.title("Revenue by Category")
plt.xlabel("Category")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    data=category_summary.sort_values("Profit", ascending=False),
    x="Category",
    y="Profit"
)

plt.title("Profit by Category")
plt.xlabel("Category")
plt.ylabel("Profit")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
product_summary = (
    df.groupby(
        ["Product_ID", "Product_Name", "Category"]
    )
    .agg(
        Total_Orders=("Order_ID", "nunique"),
        Units_Sold=("Quantity", "sum"),
        Revenue=("Net_Sales", "sum"),
        Profit=("Profit", "sum")
    )
    .reset_index()
)

product_summary["Profit_Margin"] = (
    product_summary["Profit"] /
    product_summary["Revenue"] * 100
)

product_summary.sort_values(
    "Revenue",
    ascending=False
)

In [ ]:
top_products = (
    product_summary
    .sort_values("Revenue", ascending=False)
    .head(10)
)

top_products

In [ ]:
plt.figure(figsize=(12, 6))

sns.barplot(
    data=top_products,
    x="Revenue",
    y="Product_Name"
)

plt.title("Top 10 Products by Revenue")
plt.xlabel("Revenue")
plt.ylabel("Product")
plt.tight_layout()
plt.show()

In [ ]:
top_profit_products = (
    product_summary
    .sort_values("Profit", ascending=False)
    .head(10)
)

top_profit_products

In [ ]:
top_margin_products = (
    product_summary
    .sort_values("Profit_Margin", ascending=False)
    .head(10)
)

top_margin_products[
    [
        "Product_Name",
        "Category",
        "Revenue",
        "Profit",
        "Profit_Margin"
    ]
]

In [ ]:
category_summary.to_csv(
    "../outputs/category_analysis.csv",
    index=False
)

product_summary.to_csv(
    "../outputs/product_analysis.csv",
    index=False
)

print("Product and category analysis saved successfully!")

In [ ]:
location_summary = (
    df.groupby("Location")
    .agg(
        Total_Orders=("Order_ID", "nunique"),
        Units_Sold=("Quantity", "sum"),
        Revenue=("Net_Sales", "sum"),
        Profit=("Profit", "sum")
    )
    .reset_index()
)

location_summary["Profit_Margin"] = (
    location_summary["Profit"] /
    location_summary["Revenue"] * 100
)

location_summary.sort_values(
    "Revenue",
    ascending=False
)

In [ ]:
plt.figure(figsize=(12, 6))

sns.barplot(
    data=location_summary.sort_values("Revenue", ascending=False),
    x="Revenue",
    y="Location"
)

plt.title("Revenue by Location")
plt.xlabel("Revenue")
plt.ylabel("Location")
plt.tight_layout()
plt.show()

In [ ]:
channel_summary = (
    df.groupby("Order_Channel")
    .agg(
        Total_Orders=("Order_ID", "nunique"),
        Units_Sold=("Quantity", "sum"),
        Revenue=("Net_Sales", "sum"),
        Profit=("Profit", "sum")
    )
    .reset_index()
)

channel_summary["Profit_Margin"] = (
    channel_summary["Profit"] /
    channel_summary["Revenue"] * 100
)

channel_summary

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    data=channel_summary,
    x="Order_Channel",
    y="Revenue"
)

plt.title("Revenue by Order Channel")
plt.xlabel("Order Channel")
plt.ylabel("Revenue")
plt.tight_layout()
plt.show()

In [ ]:
payment_summary = (
    df.groupby("Payment_Mode")
    .agg(
        Total_Orders=("Order_ID", "nunique"),
        Units_Sold=("Quantity", "sum"),
        Revenue=("Net_Sales", "sum"),
        Profit=("Profit", "sum")
    )
    .reset_index()
)

payment_summary["Profit_Margin"] = (
    payment_summary["Profit"] /
    payment_summary["Revenue"] * 100
)

payment_summary.sort_values(
    "Revenue",
    ascending=False
)

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    data=payment_summary.sort_values("Revenue", ascending=False),
    x="Payment_Mode",
    y="Revenue"
)

plt.title("Revenue by Payment Mode")
plt.xlabel("Payment Mode")
plt.ylabel("Revenue")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
channel_summary[
    [
        "Order_Channel",
        "Revenue",
        "Profit",
        "Profit_Margin"
    ]
].sort_values(
    "Revenue",
    ascending=False
)

In [ ]:
location_summary.to_csv(
    "../outputs/location_analysis.csv",
    index=False
)

channel_summary.to_csv(
    "../outputs/channel_analysis.csv",
    index=False
)

payment_summary.to_csv(
    "../outputs/payment_analysis.csv",
    index=False
)

print("Location, channel and payment analysis saved successfully!")

In [ ]:
return_summary = (
    df.groupby("Returned")
    .agg(
        Total_Orders=("Order_ID", "nunique"),
        Revenue=("Net_Sales", "sum"),
        Profit=("Profit", "sum")
    )
    .reset_index()
)

return_summary["Return_Rate"] = (
    return_summary["Total_Orders"] /
    return_summary["Total_Orders"].sum() * 100
)

return_summary

In [ ]:
category_returns = (
    df.groupby("Category")
    .agg(
        Total_Orders=("Order_ID", "nunique"),
        Returned_Orders=("Returned", "sum"),
        Revenue=("Net_Sales", "sum"),
        Profit=("Profit", "sum")
    )
    .reset_index()
)

category_returns["Return_Rate"] = (
    category_returns["Returned_Orders"] /
    category_returns["Total_Orders"] * 100
)

category_returns.sort_values(
    "Return_Rate",
    ascending=False
)

In [ ]:
product_returns = (
    df.groupby(
        ["Product_ID", "Product_Name", "Category"]
    )
    .agg(
        Total_Orders=("Order_ID", "nunique"),
        Returned_Orders=("Returned", "sum"),
        Revenue=("Net_Sales", "sum"),
        Returned_Revenue=(
            "Net_Sales",
            lambda x: x[df.loc[x.index, "Returned"] == 1].sum()
        ),
        Returned_Profit=(
            "Profit",
            lambda x: x[df.loc[x.index, "Returned"] == 1].sum()
        )
    )
    .reset_index()
)

product_returns["Return_Rate"] = (
    product_returns["Returned_Orders"] /
    product_returns["Total_Orders"] * 100
)

product_returns.sort_values(
    "Return_Rate",
    ascending=False
).head(10)

In [ ]:
channel_returns = (
    df.groupby("Order_Channel")
    .agg(
        Total_Orders=("Order_ID", "nunique"),
        Returned_Orders=("Returned", "sum"),
        Revenue=("Net_Sales", "sum"),
        Returned_Profit=(
            "Profit",
            lambda x: x[df.loc[x.index, "Returned"] == 1].sum()
        )
    )
    .reset_index()
)

channel_returns["Return_Rate"] = (
    channel_returns["Returned_Orders"] /
    channel_returns["Total_Orders"] * 100
)

channel_returns

In [ ]:
df["Discount_Band"] = pd.cut(
    df["Discount_Rate"],
    bins=[-0.001, 0, 0.05, 0.10, 0.15, float("inf")],
    labels=[
        "0% Discount",
        "1%-5% Discount",
        "6%-10% Discount",
        "11%-15% Discount",
        "16%+ Discount"
    ]
)

In [ ]:
discount_summary = (
    df.groupby("Discount_Band", observed=True)
    .agg(
        Total_Orders=("Order_ID", "nunique"),
        Units_Sold=("Quantity", "sum"),
        Revenue=("Net_Sales", "sum"),
        Profit=("Profit", "sum")
    )
    .reset_index()
)

discount_summary["Profit_Margin"] = (
    discount_summary["Profit"] /
    discount_summary["Revenue"] * 100
)

discount_summary

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    data=discount_summary,
    x="Discount_Band",
    y="Profit_Margin"
)

plt.title("Profit Margin by Discount Band")
plt.xlabel("Discount Band")
plt.ylabel("Profit Margin (%)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
return_summary.to_csv(
    "../outputs/return_summary.csv",
    index=False
)

category_returns.to_csv(
    "../outputs/category_returns.csv",
    index=False
)

product_returns.to_csv(
    "../outputs/product_returns.csv",
    index=False
)

channel_returns.to_csv(
    "../outputs/channel_returns.csv",
    index=False
)

discount_summary.to_csv(
    "../outputs/discount_analysis.csv",
    index=False
)

print("Return and discount analysis saved successfully!")

In [ ]:
monthly_summary = (
    df.groupby(["Year", "Month", "Month_Name"])
    .agg(
        Total_Orders=("Order_ID", "nunique"),
        Units_Sold=("Quantity", "sum"),
        Revenue=("Net_Sales", "sum"),
        Profit=("Profit", "sum")
    )
    .reset_index()
)

monthly_summary["Profit_Margin"] = (
    monthly_summary["Profit"] /
    monthly_summary["Revenue"] * 100
)

monthly_summary

In [ ]:
monthly_summary["Month_Date"] = pd.to_datetime(
    monthly_summary["Year"].astype(str)
    + "-"
    + monthly_summary["Month"].astype(str)
    + "-01"
)

monthly_summary = monthly_summary.sort_values("Month_Date")

monthly_summary.head()

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(
    monthly_summary["Month_Date"],
    monthly_summary["Revenue"],
    marker="o"
)

plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(
    monthly_summary["Month_Date"],
    monthly_summary["Profit"],
    marker="o"
)

plt.title("Monthly Profit Trend")
plt.xlabel("Month")
plt.ylabel("Profit")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(
    monthly_summary["Month_Date"],
    monthly_summary["Total_Orders"],
    marker="o"
)

plt.title("Monthly Orders Trend")
plt.xlabel("Month")
plt.ylabel("Total Orders")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
yearly_revenue = (
    monthly_summary
    .groupby("Year")["Revenue"]
    .sum()
    .reset_index()
)

yearly_revenue

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    data=yearly_revenue,
    x="Year",
    y="Revenue"
)

plt.title("Revenue Comparison: 2024 vs 2025")
plt.xlabel("Year")
plt.ylabel("Revenue")
plt.tight_layout()
plt.show()

In [ ]:
highest_revenue_month = monthly_summary.loc[
    monthly_summary["Revenue"].idxmax()
]

lowest_revenue_month = monthly_summary.loc[
    monthly_summary["Revenue"].idxmin()
]

print(
    "Highest Revenue Month:",
    highest_revenue_month["Month_Name"],
    int(highest_revenue_month["Year"]),
    f"₹{highest_revenue_month['Revenue']:,.2f}"
)

print(
    "Lowest Revenue Month:",
    lowest_revenue_month["Month_Name"],
    int(lowest_revenue_month["Year"]),
    f"₹{lowest_revenue_month['Revenue']:,.2f}"
)

In [ ]:
monthly_summary.to_csv(
    "../outputs/monthly_sales_analysis.csv",
    index=False
)

print("Monthly sales analysis saved successfully!")

In [ ]:
business_insights = pd.DataFrame({
    "Area": [
        "Overall Sales",
        "Year-over-Year",
        "Customer Analysis",
        "Category Performance",
        "Product Performance",
        "Location Performance",
        "Order Channel",
        "Returns",
        "Discounts"
    ],
    "Key_Insight": [
        "Total revenue was approximately ₹999.05M with profit of approximately ₹158.36M.",
        "Revenue remained almost stable from 2024 to 2025, while profit declined slightly.",
        "Repeat customers represented almost the entire customer base and revenue.",
        "Electronics generated the highest revenue and profit.",
        "Laptop was the highest-revenue product, while Vacuum Cleaner had one of the strongest profit margins.",
        "Ahmedabad generated the highest revenue among the analyzed locations.",
        "Online orders generated substantially more revenue than Store orders.",
        "Overall return rate was approximately 5.92%; Furniture had the highest category return rate.",
        "Higher discount bands were associated with lower profit margins."
    ]
})

business_insights

In [ ]:
business_insights.to_csv(
    "../outputs/business_insights.csv",
    index=False
)

print("Business insights saved successfully!")

In [ ]:
final_summary = pd.DataFrame({
    "Metric": [
        "Total Orders",
        "Total Customers",
        "Total Products",
        "Total Units Sold",
        "Total Revenue",
        "Total Profit",
        "Profit Margin",
        "Return Rate"
    ],
    "Value": [
        total_orders,
        total_customers,
        total_products,
        total_units,
        total_revenue,
        total_profit,
        profit_margin,
        (df["Returned"].sum() / len(df)) * 100
    ]
})

final_summary

In [ ]:
final_summary.to_csv(
    "../outputs/final_eda_summary.csv",
    index=False
)

print("Final EDA summary saved successfully!")

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(
    monthly_summary["Month_Date"],
    monthly_summary["Revenue"],
    marker="o"
)

plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig(
    "../outputs/charts/monthly_revenue_trend.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    data=category_summary.sort_values("Revenue", ascending=False),
    x="Category",
    y="Revenue"
)

plt.title("Revenue by Category")
plt.xlabel("Category")
plt.ylabel("Revenue")
plt.tight_layout()

plt.savefig(
    "../outputs/charts/category_revenue.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

sns.barplot(
    data=top_products,
    x="Revenue",
    y="Product_Name"
)

plt.title("Top 10 Products by Revenue")
plt.xlabel("Revenue")
plt.ylabel("Product")
plt.tight_layout()

plt.savefig(
    "../outputs/charts/top_10_products.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    data=discount_summary,
    x="Discount_Band",
    y="Profit_Margin"
)

plt.title("Profit Margin by Discount Band")
plt.xlabel("Discount Band")
plt.ylabel("Profit Margin (%)")
plt.xticks(rotation=30)
plt.tight_layout()

plt.savefig(
    "../outputs/charts/discount_vs_profit_margin.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# FINAL EDA SUMMARY
# ==========================================

print("========== RETAIL SALES EDA SUMMARY ==========")

print(f"Total Orders       : {total_orders:,}")
print(f"Total Customers    : {total_customers:,}")
print(f"Total Products     : {total_products:,}")
print(f"Total Units Sold   : {total_units:,}")
print(f"Total Revenue      : ₹{total_revenue:,.2f}")
print(f"Total Profit       : ₹{total_profit:,.2f}")
print(f"Profit Margin      : {profit_margin:.2f}%")

# Return rate
overall_return_rate = df["Returned"].mean() * 100

print(f"Return Rate        : {overall_return_rate:.2f}%")

# Best category
best_category = category_summary.loc[
    category_summary["Revenue"].idxmax(), "Category"
]

# Best product
best_product = product_summary.loc[
    product_summary["Revenue"].idxmax(), "Product_Name"
]

# Best location
best_location = location_summary.loc[
    location_summary["Revenue"].idxmax(), "Location"
]

# Best channel
best_channel = channel_summary.loc[
    channel_summary["Revenue"].idxmax(), "Order_Channel"
]

print(f"Top Category       : {best_category}")
print(f"Top Product        : {best_product}")
print(f"Top Location       : {best_location}")
print(f"Top Channel        : {best_channel}")

In [ ]:
# ==========================================
# REVENUE BY CATEGORY
# ==========================================

plt.figure(figsize=(10, 6))

plt.bar(
    category_summary["Category"],
    category_summary["Revenue"]
)

plt.title("Revenue by Category")
plt.xlabel("Category")
plt.ylabel("Revenue")
plt.xticks(rotation=30)

plt.tight_layout()

plt.savefig(
    "../outputs/charts/revenue_by_category.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# PROFIT BY CATEGORY
# ==========================================

plt.figure(figsize=(10, 6))

plt.bar(
    category_summary["Category"],
    category_summary["Profit"]
)

plt.title("Profit by Category")
plt.xlabel("Category")
plt.ylabel("Profit")
plt.xticks(rotation=30)

plt.tight_layout()

plt.savefig(
    "../outputs/charts/profit_by_category.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# TOP 10 PRODUCTS BY REVENUE
# ==========================================

top_10_products = (
    product_summary
    .sort_values("Revenue", ascending=False)
    .head(10)
    .sort_values("Revenue")
)

plt.figure(figsize=(10, 6))

plt.barh(
    top_10_products["Product_Name"],
    top_10_products["Revenue"]
)

plt.title("Top 10 Products by Revenue")
plt.xlabel("Revenue")
plt.ylabel("Product")

plt.tight_layout()

plt.savefig(
    "../outputs/charts/top_10_products_revenue.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()